<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
DRIVE_OUTPUTS = "/content/drive/MyDrive/work/outputs"
DRIVE_FIGURES = "/content/drive/MyDrive/work/figures"
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
os.makedirs(DRIVE_FIGURES, exist_ok=True)

In [3]:
DRIVE_OUTPUTS = "/content/drive/MyDrive/work/outputs"

REQUIRED_OUTPUTS = [
    f"{DRIVE_OUTPUTS}/baseline_action_score.csv",
    f"{DRIVE_OUTPUTS}/model_vs_baseline.csv",
    f"{DRIVE_OUTPUTS}/split_before_after.csv",
    f"{DRIVE_OUTPUTS}/content_action_queue.csv",
    f"{DRIVE_OUTPUTS}/playbook_metrics.json",
]
print(os.listdir("/content/drive/MyDrive/work/outputs"))
# ...and every read/write inside the notebook uses DRIVE_OUTPUTS the same way

['content_action_queue.csv', 'playbook_metrics.json', 'capstone_metrics.json', '_train_idx.npy', '_test_idx.npy', 'model_vs_baseline.csv', 'baseline_action_score.csv', 'split_before_after.csv']


# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

FlyRank builds content as infrastructure — researching, writing, and publishing pages directly into client websites, then watching search performance and optimizing, largely through algorithms rather than manual review. Content that ranks well quietly decays over time: rankings slip, clicks fall, and teams often notice only after the damage is done. FlyRank's product already answers a coarser version of this with hand-written flags — a health score, quick-win tags, needs-attention markers. This project's hand-written baseline plays that same role for this exercise — a transparent stand-in to measure against, not FlyRank's actual production system.





**Question:** Which pages should a content team review first for refresh, given limited reviewer time?

**Decision this supports:** which pages get a human content reviewer's attention this week, out of thousands of candidates. The action taken is a human review — update, expand, fix metadata, or leave alone — never an automated edit.

**Cost of a wrong call, and why this matters for the design:** a false positive (flagging a healthy page) costs bounded reviewer time. A false negative (missing a genuinely declining page) is invisible and compounds silently — the page keeps losing visibility unattended. This asymmetry is why the deliverable is a **ranked list with reason codes**, not a hard yes/no cutoff: it lets a human apply judgment rather than treating the model as automatic.

**Target:** `is_declining = (trend_direction == "down")` — an observed outcome, computed upstream from a measured 30-day-over-30-day trend, not a rule invented for this project (see `w02_ml_task_framing.ipynb`, `w03_data_contract.ipynb`).

In [4]:
import os
import json
import pandas as pd
import numpy as np

# This notebook assumes it is run from the repository root (as w01-w07 do).
# In Colab: run the standard clone/chdir cell from any other notebook in this repo first,
# or simply open this notebook via the badge above, which starts you in /content and
# you then `os.chdir` into the cloned repo exactly as the other notebooks do.

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

RANDOM_SEED = 42
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), (
    "Run from the repository root. If in Colab, clone the repo and os.chdir into it first, "
    "exactly as in w01_research_question.ipynb."
)

DRIVE_OUTPUTS = "/content/drive/MyDrive/work/outputs"

REQUIRED_OUTPUTS = [
    f"{DRIVE_OUTPUTS}/baseline_action_score.csv",
    f"{DRIVE_OUTPUTS}/model_vs_baseline.csv",
    f"{DRIVE_OUTPUTS}/split_before_after.csv",
    f"{DRIVE_OUTPUTS}/content_action_queue.csv",
    f"{DRIVE_OUTPUTS}/playbook_metrics.json",
]

missing = [p for p in REQUIRED_OUTPUTS if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Missing required upstream outputs: " + ", ".join(missing) +
        "\nRun w04 -> w05 -> w06 -> w07 top-to-bottom first. This notebook consolidates "
        "their outputs; it does not regenerate them."
    )

print("All required upstream outputs found:")
for p in REQUIRED_OUTPUTS:
    print("  ", p)

df_raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
base_rate_full = (df_raw["trend_direction"] == "down").mean()
print("All required upstream outputs found.")
print(f"Dataset: {len(df_raw):,} rows, {df_raw['client_id'].nunique()} clients")
print(f"Full-dataset base rate (is_declining): {base_rate_full:.3f}")

All required upstream outputs found:
   /content/drive/MyDrive/work/outputs/baseline_action_score.csv
   /content/drive/MyDrive/work/outputs/model_vs_baseline.csv
   /content/drive/MyDrive/work/outputs/split_before_after.csv
   /content/drive/MyDrive/work/outputs/content_action_queue.csv
   /content/drive/MyDrive/work/outputs/playbook_metrics.json
All required upstream outputs found.
Dataset: 30,000 rows, 32 clients
Full-dataset base rate (is_declining): 0.542


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank ML Internship starter dataset, `data/raw/content_refresh_anonymized.csv` — 30,000 pseudonymized content items across 32 clients, one row per `content_id`. This is the small starter slice, not the full warehouse (`hf://datasets/FlyRank/internship-warehouse`); no calendar dates exist in this file, only trailing windows (`_90d`, `_last_30d`, `_prev_30d`) as of an implicit snapshot.

**Excluded, with why:** `impressions_last_30d` and `impressions_prev_30d` can directly derive the decline label and are excluded from every feature set as a confirmed leakage risk (verified in `w03_data_contract.ipynb` and re-confirmed below). `client_id` is used only for grouped train/test splitting, never as a feature, and never appears in any printed sample or written output.

In [5]:
leakage_cols = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"}
feature_cols = ["impressions_90d", "ctr", "avg_position", "engagement_rate", "scroll_rate",
                "days_since_last_update", "content_age_days", "word_count",
                "freshness_tier", "days_with_impressions"]

print("Feature columns:", feature_cols)
print("Overlap with leakage-risk columns (must be empty):", set(feature_cols) & leakage_cols)
assert len(set(feature_cols) & leakage_cols) == 0

print(f"\navg_position == 0 ('no data', not rank zero): {(df_raw['avg_position']==0).sum():,} rows")
print(f"Missing scroll_rate: {df_raw['scroll_rate'].isnull().sum():,} rows")
print(f"Missing word_count: {df_raw['word_count'].isnull().sum():,} rows")
print("\nclient_id is present only for grouped splitting, confirmed absent from every written output below.")

Feature columns: ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'days_since_last_update', 'content_age_days', 'word_count', 'freshness_tier', 'days_with_impressions']
Overlap with leakage-risk columns (must be empty): set()

avg_position == 0 ('no data', not rank zero): 1,205 rows
Missing scroll_rate: 125 rows
Missing word_count: 7,699 rows

client_id is present only for grouped splitting, confirmed absent from every written output below.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

- **Label:** `is_declining = (trend_direction == "down")`, an observed outcome, not a hand-defined rule.
- **Baseline:** a plain-language, non-fitted rule — review a page if it's stale (≥180 days since update), still visible (≥500 impressions/90d), and sits in a recoverable position band (10th–50th avg. position). Frozen once model work began (`w04_baseline_score.ipynb`).
- **Validation:** 80/20 `GroupShuffleSplit` by `client_id`, seed 42, reused identically across the baseline, both trained models, and all downstream audit/playbook work (`w05_model.ipynb`).
- **Leakage checks:** a positive control deliberately reintroducing `trend_pct` (the value the label is computed from) produced a large, detectable AUC jump, confirming the audit harness itself works (`w06_validation_audit.ipynb`).

In [6]:
split_audit = pd.read_csv(f"{DRIVE_OUTPUTS}/split_before_after.csv")
before_mean, before_sd = split_audit["before_random_split"].mean(), split_audit["before_random_split"].std()
after_mean, after_sd = split_audit["after_grouped_split"].mean(), split_audit["after_grouped_split"].std()

print("=== Validation design check: naive random split vs. honest client-grouped split ===")
print(f"Naive random split (precision@50):    mean={before_mean:.3f}  sd={before_sd:.3f}")
print(f"Honest client-grouped (precision@50):  mean={after_mean:.3f}  sd={after_sd:.3f}")
print(f"Memorization gap: {before_mean - after_mean:+.3f}")
print("\nThis measured gap is why the client-grouped split is used everywhere in this project, not a random row split.")

=== Validation design check: naive random split vs. honest client-grouped split ===
Naive random split (precision@50):    mean=0.828  sd=0.026
Honest client-grouped (precision@50):  mean=0.735  sd=0.064
Memorization gap: +0.093

This measured gap is why the client-grouped split is used everywhere in this project, not a random row split.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [7]:
comparison = pd.read_csv(f"{DRIVE_OUTPUTS}/model_vs_baseline.csv")
print("=== Model vs. baseline, precision@K, identical held-out client-grouped test split ===")
print(comparison.to_string(index=False))

lr_beats_base = (comparison["logistic_regression"] > comparison["base_rate"]).all()
lr_beats_rule = (comparison["logistic_regression"] > comparison["baseline_rule"]).all()
rf_loses_low_k = (comparison.loc[comparison["K"].isin([20, 50]), "random_forest"]
                   < comparison.loc[comparison["K"].isin([20, 50]), "base_rate"]).all()

print(f"\nLogistic Regression beats base rate at every K: {lr_beats_base}")
print(f"Logistic Regression beats the rule baseline at every K: {lr_beats_rule}")
print(f"Random Forest underperforms base rate at K=20 and K=50: {rf_loses_low_k}")

# Soft check, not a hard assert -- matches w05_model.ipynb's own self-check design.
# GroupShuffleSplit's exact client assignment can vary slightly across sklearn/numpy
# versions even with the same random_state, so this is a NOTE to re-verify, not a crash.
if lr_beats_base and lr_beats_rule:
    print("\nDecision confirmed by this run: Logistic Regression ships. Random Forest is a documented negative result, kept in the table, not shipped -- per 'add complexity only when the comparison earns it.'")
else:
    print("\nNOTE: this run's numbers differ from a prior run (possibly due to sklearn/numpy version")
    print("differences affecting GroupShuffleSplit's exact client assignment). Re-check the 'shipped")
    print("model' decision against THIS run's table before treating it as final, per the same honesty")
    print("standard used throughout w05_model.ipynb.")

=== Model vs. baseline, precision@K, identical held-out client-grouped test split ===
  K  base_rate  baseline_rule  logistic_regression  random_forest
 20      0.511           0.45                 0.75           0.40
 50      0.511           0.62                 0.74           0.48
100      0.511           0.61                 0.73           0.53

Logistic Regression beats base rate at every K: True
Logistic Regression beats the rule baseline at every K: True
Random Forest underperforms base rate at K=20 and K=50: True

Decision confirmed by this run: Logistic Regression ships. Random Forest is a documented negative result, kept in the table, not shipped -- per 'add complexity only when the comparison earns it.'


## 5. Limitations

*What this work cannot claim.*

In [8]:
with open(f"{DRIVE_OUTPUTS}/playbook_metrics.json") as f:


    playbook_metrics = json.load(f)

print("Precision@K describes ranking strength at the TOP of the list, not a blanket classifier.")
print("Base rate this playbook was built on:", playbook_metrics["base_rate_test_split"])
print("Visibility floor (pages below this are simply unmeasured, not 'not declining'):",
      playbook_metrics["visibility_floor_impressions_90d"], "impressions/90d")
print("\nSplit variance: precision@50 sd across repeated client-grouped splits was", round(after_sd, 3),
      "-- a single reported number understates this variability.")

Precision@K describes ranking strength at the TOP of the list, not a blanket classifier.
Base rate this playbook was built on: 0.511
Visibility floor (pages below this are simply unmeasured, not 'not declining'): 500 impressions/90d

Split variance: precision@50 sd across repeated client-grouped splits was 0.064 -- a single reported number understates this variability.


**What this project can say:** the ranked queue is observed, measured, decision-support for a human reviewer — validated on held-out clients never seen in training.

**What this project cannot say:** that reviewing or refreshing a flagged page *causes* it to recover (no causal experiment exists in this data); anything about Google's actual ranking algorithm; that results here transfer unchecked to the full warehouse release, since this is a 30,000-row starter slice; that the model is reliably accurate in every traffic tier — accuracy was measurably weakest (~0.49) in the highest-traffic tier, exactly where a wrong call is most costly.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [9]:
queue = pd.read_csv(f"{DRIVE_OUTPUTS}/content_action_queue.csv")
assert "client_id" not in queue.columns, "client_id must never appear in a shared/exported file."

archetype_summary = queue.groupby("action").agg(
    n=("content_id", "size"),
    observed_decline_rate=("trend_direction", lambda s: (s == "down").mean())
).round(3).sort_values("observed_decline_rate", ascending=False)

eligible_base_rate = (queue["trend_direction"] == "down").mean()
print("=== Archetype summary (eligible queue) ===")
print(archetype_summary)
print(f"\nEligible-population base rate: {eligible_base_rate:.3f}")

print("\n=== Top 10 pages, ranked queue ===")
print(queue.sort_values("model_score", ascending=False)
      [["content_id", "model_score", "action", "reason_code"]]
      .head(10).to_string(index=False))

=== Archetype summary (eligible queue) ===
            n  observed_decline_rate
action                              
Fix CTR  5893                  0.665
Support  1371                  0.573
Refresh  2445                  0.562
Monitor  7017                  0.554

Eligible-population base rate: 0.596

=== Top 10 pages, ranked queue ===
          content_id  model_score  action     reason_code
content_8f3b70ede1bc     0.869499 Fix CTR         fix_ctr
content_8ede62882d0b     0.867625 Fix CTR         fix_ctr
content_b08562686d22     0.865670 Fix CTR         fix_ctr
content_67a766790dd2     0.848664 Fix CTR         fix_ctr
content_26d48a980581     0.844939 Fix CTR         fix_ctr
content_2bc3b7c8b3d9     0.842821 Monitor no_flag_monitor
content_c94a53e3bfb8     0.842707 Monitor no_flag_monitor
content_3ff647c911d0     0.834366 Monitor no_flag_monitor
content_0361d8df96e6     0.830182 Fix CTR         fix_ctr
content_58c8c8598fd5     0.826378 Fix CTR         fix_ctr


**Ranked recommendations for a content team:**

1. Prioritize the Logistic Regression–ranked queue over the hand-written rule for weekly review — it delivers meaningfully higher, validated precision without added model complexity cost.
2. Route pages by archetype, not a single "refresh" bucket — Fix CTR, Refresh, and Support imply different editorial actions; only Fix CTR currently shows a decline rate clearly above the eligible-population base rate.
3. Treat the top of the queue as the trustworthy core — precision degrades past the top of the ranking.
4. Do not automate content edits, deletions, or client-facing forecasts — this system narrows a human's queue, it does not replace their judgment.
5. Re-validate if a fresh sample's precision@50 drops more than one sd below the historical grouped-split mean, if the base decline rate shifts more than 10 points, or at minimum quarterly.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

order = archetype_summary.index.tolist()
rates = archetype_summary["observed_decline_rate"].tolist()
ns = archetype_summary["n"].tolist()
labels = [f"{name}\nn={n:,}" for name, n in zip(order, ns)]

fig, ax = plt.subplots(figsize=(7.2, 4.2))
bars = ax.bar(order, rates, color="#2a78d6", width=0.55, zorder=3)
ax.axhline(eligible_base_rate, color="#C97A2C", linestyle="--", linewidth=1.4,
           label=f"base rate ({eligible_base_rate:.3f})")
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, rate + 0.012, f"{rate:.3f}", ha="center", fontsize=10)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(labels)
ax.set_ylabel("Observed decline rate")
ax.set_title("Observed decline rate by archetype (Logistic Regression scoring)")
ax.legend()
plt.tight_layout()
os.makedirs("work/figures", exist_ok=True)
plt.savefig("work/figures/archetype_decline_rate.png", dpi=150)
plt.show()
print("Saved work/figures/archetype_decline_rate.png -- copy this into docs/images/ for the deployed paper.")

# --- Consolidated capstone metrics JSON, for the paper and as a reproducibility artifact ---
capstone_metrics = {
    "capstone": {
        "lane": "Refresh / Content Opportunity Scoring",
        "n_rows": int(len(df_raw)),
        "n_clients": int(df_raw["client_id"].nunique()),
        "base_rate_full_dataset": round(float(base_rate_full), 3),
    },
    "model_comparison": {
        "precision_at_k": comparison.set_index("K").to_dict(orient="index"),
        "shipped_model": "logistic_regression",
    },
    "validation_audit": {
        "random_row_split": {"mean": round(float(before_mean), 3), "sd": round(float(before_sd), 3)},
        "client_grouped_split": {"mean": round(float(after_mean), 3), "sd": round(float(after_sd), 3)},
        "memorization_gap": round(float(before_mean - after_mean), 3),
    },
    "action_playbook": {
        "eligible_base_rate": round(float(eligible_base_rate), 3),
        "precision_at_k": playbook_metrics["precision_at_k"],
        "archetype_summary": archetype_summary.to_dict(orient="index"),
    },
    "source_notebooks": [
        "w01_research_question.ipynb", "w02_ml_task_framing.ipynb", "w03_data_contract.ipynb",
        "w04_baseline_score.ipynb", "w05_model.ipynb", "w06_validation_audit.ipynb",
        "w07_action_playbook.ipynb", "capstone.ipynb",
    ],
}
with open(f"{DRIVE_OUTPUTS}/capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2)
print(f"Wrote {DRIVE_OUTPUTS}/capstone_metrics.json")

Saved work/figures/archetype_decline_rate.png -- copy this into docs/images/ for the deployed paper.
Wrote /content/drive/MyDrive/work/outputs/capstone_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


In [11]:
# Enforced self-check for the parts that should always hold regardless of split variance.
assert os.path.exists(f"{DRIVE_OUTPUTS}/capstone_metrics.json")
assert "client_id" not in queue.columns
assert lr_beats_base, "Logistic Regression should always beat the base rate -- this one should never fail."
print("Self-check PASSED: capstone_metrics.json written, no client_id leaked, LR beats base rate.")
if not lr_beats_rule:
    print("NOTE: LR did not beat the rule baseline at every K in THIS run -- see Section 4's note above "
          "before finalizing the 'shipped model' claim in your deployed paper.")

Self-check PASSED: capstone_metrics.json written, no client_id leaked, LR beats base rate.


### ML-12 — 5-minute demo outline

1. **(0:00–0:45) The problem.** A content team has thousands of pages and no way to know which ones need attention first. Show the scale: 30,000 pages, 32 clients, ~55% currently declining by observed trend.
2. **(0:45–1:30) The naive approach and why it's not enough.** Show the hand-written rule from `w04` — only 14 of 30,000 pages ever clear every threshold. Too restrictive to be useful at scale.
3. **(1:30–2:30) The model.** Walk through the precision@K table from Section 4: Logistic Regression beats the rule and the base rate at every K; Random Forest, despite more complexity, does not — and was rejected for exactly that reason.
4. **(2:30–3:15) Why the split matters.** Show the random-vs-grouped split gap from Section 3 (~0.09) — a plain random split would have overstated performance by memorizing clients.
5. **(3:15–4:15) The deliverable.** Show the ranked, reason-coded queue and the archetype chart — this is what a content strategist actually opens Monday morning.
6. **(4:15–5:00) Limitations, stated out loud.** No causal claim, no Google-ranking claim, weakest exactly in the highest-traffic tier — named directly, not hidden.

### ML-12 — Social-post cut

> Content teams can't review every page. So I built a system that ranks which pages to check first — and proved it beats a hand-written rule *and* a fancier Random Forest model on held-out clients (precision@50 of 0.74 vs. a 0.51 base rate). The honest part: I also show where it's weakest, not just where it wins. 🧠📈 #MachineLearning #ContentStrategy

### ML-12 — 3-sentence employer-facing summary

I built a page-prioritization system for content teams that ranks thousands of pages by refresh opportunity, validated against a hand-written baseline on client-held-out data rather than a convenient random split. The shipped model (Logistic Regression) beat both the baseline rule and a more complex Random Forest at every evaluated cutoff, and I documented the negative result on Random Forest rather than omitting it. The final deliverable is a reason-coded, ranked action queue plus a deployed research paper covering methodology, honest limitations, and decision-support recommendations.

In [12]:
import shutil
shutil.copytree(DRIVE_OUTPUTS, "work/outputs", dirs_exist_ok=True)
shutil.copytree(DRIVE_FIGURES, "work/figures", dirs_exist_ok=True)

'work/figures'